# NeighborhoodCollection population feature space

This notebook creates a `NeighborhoodCollection`, calculates a neighborhood-by-population feature space, and stores the result as a MuData modality. It downloads an example `.h5ad` from the Broad Institute Celldega supporting-data repository on Hugging Face at runtime.

In [ ]:
from pathlib import Path
from urllib.parse import quote

import requests
import scanpy as sc

import celldega as dega

In [ ]:
REPO_ID = "broadinstitute/Celldega_Supporting_Data"
REVISION = "main"
CACHE_DIR = Path("data/celldega_supporting_data")
H5AD_PATH = "Xenium_Prime_Human_Skin_FFPE_outs.h5ad"


def download_repo_file(repo_id, repo_path, cache_dir, revision="main"):
    cache_dir.mkdir(parents=True, exist_ok=True)
    local_path = cache_dir / Path(repo_path).name
    if local_path.exists():
        return local_path

    encoded_path = quote(repo_path)
    url = f"https://huggingface.co/datasets/{repo_id}/resolve/{revision}/{encoded_path}"
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with local_path.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
    return local_path


local_h5ad = download_repo_file(REPO_ID, H5AD_PATH, CACHE_DIR, REVISION)
adata = sc.read_h5ad(local_h5ad)
adata

In [ ]:
def ensure_spatial_coordinates(adata):
    if "spatial" in adata.obsm:
        return

    coordinate_pairs = [
        ("x_centroid", "y_centroid"),
        ("x_location", "y_location"),
        ("x", "y"),
        ("X_centroid", "Y_centroid"),
    ]
    for x_col, y_col in coordinate_pairs:
        if x_col in adata.obs and y_col in adata.obs:
            adata.obsm["spatial"] = adata.obs[[x_col, y_col]].to_numpy()
            return

    raise ValueError("The AnnData needs adata.obsm['spatial'] or x/y centroid columns.")


ensure_spatial_coordinates(adata)
population_col = "leiden"
if population_col not in adata.obs:
    raise ValueError(f"Expected adata.obs[{population_col!r}] in this example.")
population_col

In [ ]:
gdf_hex = dega.nbhd.generate_hextile(adata, diameter=100)
gdf_hex.head()

In [ ]:
nbhd = dega.nbhd.NeighborhoodCollection(
    gdf=gdf_hex,
    nbhd_type="hextile",
    adata=adata,
)

nbhd.calc_nbhd_by_pop(
    category=population_col,
    output="counts",
    min_cells=5,
)

population = nbhd.mod["population"]
population

In [ ]:
population_for_clustering = population.copy()

if population_for_clustering.n_obs > 2:
    sc.pp.normalize_total(population_for_clustering, target_sum=1)
    sc.pp.neighbors(
        population_for_clustering,
        n_neighbors=min(15, population_for_clustering.n_obs - 1),
        metric="cosine",
    )
    sc.tl.leiden(population_for_clustering, key_added="nbhd_leiden", resolution=0.8)
else:
    population_for_clustering.obs["nbhd_leiden"] = "0"

nbhd_leiden = population_for_clustering.obs["nbhd_leiden"].astype("category")
population.obs["nbhd_leiden"] = nbhd_leiden
nbhd.obs["nbhd_leiden"] = nbhd_leiden.reindex(nbhd.obs.index).astype("category")

cluster_values = nbhd.obs["nbhd_leiden"].astype(str)
palette = sc.pl.palettes.default_20
cluster_colors = {
    cluster: palette[i % len(palette)]
    for i, cluster in enumerate(sorted(cluster_values.dropna().unique()))
}
nbhd.obs["cat"] = cluster_values
nbhd.obs["color"] = cluster_values.map(cluster_colors).fillna("#4f80ff")

nbhd.obs[["nbhd_leiden", "color"]].head()

In [ ]:
nbhd.geometry.plot()

In [ ]:
nbhd.collection_type

In [ ]:
nbhd.mod

The `population` modality is stored inside the same collection object. The canonical observation axis is `nbhd.obs`, while `nbhd.mod["population"].X` is the clusterable neighborhood-by-population matrix.

In [ ]:
mat = dega.clust.Matrix(
    population,
    row_attr=[population_col],
    col_attr=["nbhd_leiden"],
)
mat.row_entity, mat.col_entity

In [ ]:
mat.cluster()

In [ ]:
cgm = dega.viz.Clustergram(matrix=mat)
cgm

In [ ]:
base_url = 'https://raw.githubusercontent.com/broadinstitute/celldega_Xenium_Prime_Human_Skin_FFPE_outs/main/Xenium_Prime_Human_Skin_FFPE_outs'

In [ ]:
nbhd.obs[["neighborhood_id", "nbhd_leiden", "cat", "color"]].head()

In [ ]:
nbhd.geometry.head()

In [ ]:
landscape = dega.viz.Landscape(
    technology='Xenium', 
    base_url = base_url,
    height=550,
    adata=adata,
    nbhd=nbhd
)

In [ ]:
dega.viz.landscape_clustergram(landscape, cgm)